# 🔧 Class 04 — Feature Engineering & Normalization

## 📌 ভূমিকা (Introduction)

এই নোটবুকে আমরা **Mobile Price Dataset** নিয়ে কাজ করব এবং শিখব দুটি অত্যন্ত গুরুত্বপূর্ণ Machine Learning পূর্বপ্রস্তুতি ধাপ:

**Feature Engineering & Normalization** ✅

### আমাদের লক্ষ্য (Our Task):
- Dataset লোড করে Explore করা (Null Check, Correlation, Statistics)
- **Normalization** কী এবং কেন করতে হয় তা বোঝা
- **Min-Max Scaling** ও **Standard Scaler** দিয়ে Data Normalize করা
- **Feature Engineering** — কোন Column Independent (X) আর কোনটি Dependent (Y) তা বের করা
- **Chi-Square Test** দিয়ে সবচেয়ে গুরুত্বপূর্ণ Feature গুলো Select করা

---

### 🧠 Normalization কী?

Real-world Dataset-এ বিভিন্ন Column-এর মানের পরিসর (Range) একদম আলাদা হয়। যেমন:
- Battery Power: 500 – 2000
- RAM: 256 – 3998
- Clock Speed: 0.5 – 3.0

এই বিশাল পার্থক্যের কারণে ML Model ভুলভাবে বড় সংখ্যার Column-কে বেশি গুরুত্বপূর্ণ মনে করে। **Normalization** সব Column-এর মানকে একটি নির্দিষ্ট Range-এ নিয়ে আসে, যাতে সব Feature সমান গুরুত্ব পায়।

### 🧠 Feature Engineering কী?

Feature Engineering হলো Raw Data থেকে **সবচেয়ে কার্যকর Feature (Column) বাছাই করার** প্রক্রিয়া। সব Feature সমান গুরুত্বপূর্ণ নয় — কিছু Feature Target Variable-এর উপর বেশি প্রভাব রাখে, কিছু প্রায় কোনো প্রভাবই রাখে না। অপ্রয়োজনীয় Feature বাদ দিলে:
- মডেল দ্রুত Train হয়
- Overfitting কমে
- Accuracy বাড়ে

---

## 🐼 প্রয়োজনীয় Library Import

In [3]:
import pandas as pd

## 📂 Dataset লোড করা

Mobile Price Dataset লোড করা হচ্ছে। এই Dataset-এ মোবাইলের বিভিন্ন Specification (RAM, Battery, Camera ইত্যাদি) এবং সেই অনুযায়ী Price Range (0, 1, 2, 3) দেওয়া আছে।

> ⚠️ **সাধারণ ভুল:** Colab-এ ফাইল আপলোড না করে রান করলে `FileNotFoundError` আসবে। বাম পাশের **Files** আইকনে ক্লিক করে আগে ফাইলটি আপলোড করে নিতে হবে।

In [4]:
dataset=pd.read_csv('/content/mobile_dataset.csv')

## 👀 সম্পূর্ণ Dataset দেখা

In [5]:
dataset

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,794,1,0.5,1,0,1,2,0.8,106,6,...,1222,1890,668,13,4,19,1,1,0,0
1996,1965,1,2.6,1,0,0,39,0.2,187,4,...,915,1965,2032,11,10,16,1,1,1,2
1997,1911,0,0.9,1,1,1,36,0.7,108,8,...,868,1632,3057,9,1,5,1,1,0,3
1998,1512,0,0.9,0,4,1,46,0.1,145,5,...,336,670,869,18,10,19,1,1,1,0


## 🔝 প্রথম ৫টি Row দেখা — `head()`

`head()` দিয়ে Dataset-এর গঠন, Column-এর নাম এবং ডেটার ধরন সম্পর্কে প্রাথমিক ধারণা পাওয়া যায়।

In [6]:
dataset.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


## ❓ Null Value চেক — `isnull().sum()`

ML Model Train করার আগে অবশ্যই Missing Value চেক করতে হবে। যদি Null থাকে, তাহলে আগে সেটা Handle করতে হবে (Imputation বা Drop)।

> 💡 যদি সব Column-এ `0` দেখায়, তাহলে Dataset সম্পূর্ণ Clean — সরাসরি পরের ধাপে যাওয়া যাবে।

In [7]:
dataset.isnull().sum()

,0
battery_power,0
blue,0
clock_speed,0
dual_sim,0
fc,0
four_g,0
int_memory,0
m_dep,0
mobile_wt,0
n_cores,0


## 🔗 Correlation দেখা — `corr()`

প্রতিটি Column-এর সাথে অন্য Column-এর সম্পর্ক দেখা হচ্ছে। এতে বোঝা যাবে কোন Feature গুলো Target Variable (`price_range`) এর সাথে বেশি সম্পর্কিত।

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \cdot \sum (y_i - \bar{y})^2}}$$

| Correlation মান | অর্থ |
|---|---|
| **+1 এর কাছে** | Strong Positive সম্পর্ক |
| **0 এর কাছে** | কোনো সম্পর্ক নেই |
| **-1 এর কাছে** | Strong Negative সম্পর্ক |

In [8]:
dataset.corr()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
battery_power,1.000000,0.011252,0.011482,-0.041847,0.033334,0.015665,-0.004004,0.034085,0.001844,-0.029727,...,0.014901,-0.008402,-0.000653,-0.029959,-0.021421,0.052510,0.011522,-0.010516,-0.008343,0.200723
blue,0.011252,1.000000,0.021419,0.035198,0.003593,0.013443,0.041177,0.004049,-0.008605,0.036161,...,-0.006872,-0.041533,0.026351,-0.002952,0.000613,0.013934,-0.030236,0.010061,-0.021863,0.020573
clock_speed,0.011482,0.021419,1.000000,-0.001315,-0.000434,-0.043073,0.006545,-0.014364,0.012350,-0.005724,...,-0.014523,-0.009476,0.003443,-0.029078,-0.007378,-0.011432,-0.046433,0.019756,-0.024471,-0.006606
dual_sim,-0.041847,0.035198,-0.001315,1.000000,-0.029123,0.003187,-0.015679,-0.022142,-0.008979,-0.024658,...,-0.020875,0.014291,0.041072,-0.011949,-0.016666,-0.039404,-0.014008,-0.017117,0.022740,0.017444
fc,0.033334,0.003593,-0.000434,-0.029123,1.000000,-0.016560,-0.029133,-0.001791,0.023618,-0.013356,...,-0.009990,-0.005176,0.015099,-0.011014,-0.012373,-0.006829,0.001793,-0.014828,0.020085,0.021998
four_g,0.015665,0.013443,-0.043073,0.003187,-0.016560,1.000000,0.008690,-0.001823,-0.016537,-0.029706,...,-0.019236,0.007448,0.007313,0.027166,0.037005,-0.046628,0.584246,0.016758,-0.017620,0.014772
int_memory,-0.004004,0.041177,0.006545,-0.015679,-0.029133,0.008690,1.000000,0.006886,-0.034214,-0.028310,...,0.010441,-0.008335,0.032813,0.037771,0.011731,-0.002790,-0.009366,-0.026999,0.006993,0.044435
m_dep,0.034085,0.004049,-0.014364,-0.022142,-0.001791,-0.001823,0.006886,1.000000,0.021756,-0.003504,...,0.025263,0.023566,-0.009434,-0.025348,-0.018388,0.017003,-0.012065,-0.002638,-0.028353,0.000853
mobile_wt,0.001844,-0.008605,0.012350,-0.008979,0.023618,-0.016537,-0.034214,0.021756,1.000000,-0.018989,...,0.000939,0.000090,-0.002581,-0.033855,-0.020761,0.006209,0.001551,-0.014368,-0.000409,-0.030302
n_cores,-0.029727,0.036161,-0.005724,-0.024658,-0.013356,-0.029706,-0.028310,-0.003504,-0.018989,1.000000,...,-0.006872,0.024480,0.004868,-0.000315,0.025826,0.013148,-0.014733,0.023774,-0.009964,0.004399


## 📊 Statistical Summary — `describe()`

`describe()` সব Numerical Column এর গুরুত্বপূর্ণ Statistics দেখায়:

| Statistics | অর্থ |
|---|---|
| `count` | কতটি Non-Null মান আছে |
| `mean` | গড় মান |
| `std` | Standard Deviation (মানগুলো গড় থেকে কতটা দূরে) |
| `min / max` | সর্বনিম্ন ও সর্বোচ্চ মান |
| `25%, 50%, 75%` | Quartile মান |

> 💡 এখানে বিভিন্ন Column-এর Range দেখো — যেমন RAM এর মান 256 থেকে 3998 পর্যন্ত, কিন্তু Clock Speed 0.5 থেকে 3 পর্যন্ত। এই বিশাল পার্থক্যই Normalization করার প্রয়োজনীয়তা বুঝিয়ে দেয়।

In [9]:
dataset.describe()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
count,2000.000000,2000.0000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,...,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,1238.518500,0.4950,1.522250,0.509500,4.309500,0.521500,32.046500,0.501750,140.249000,4.520500,...,645.108000,1251.515500,2124.213000,12.306500,5.767000,11.011000,0.761500,0.503000,0.507000,1.500000
std,439.418206,0.5001,0.816004,0.500035,4.341444,0.499662,18.145715,0.288416,35.399655,2.287837,...,443.780811,432.199447,1084.732044,4.213245,4.356398,5.463955,0.426273,0.500116,0.500076,1.118314
min,501.000000,0.0000,0.500000,0.000000,0.000000,0.000000,2.000000,0.100000,80.000000,1.000000,...,0.000000,500.000000,256.000000,5.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000
25%,851.750000,0.0000,0.700000,0.000000,1.000000,0.000000,16.000000,0.200000,109.000000,3.000000,...,282.750000,874.750000,1207.500000,9.000000,2.000000,6.000000,1.000000,0.000000,0.000000,0.750000
50%,1226.000000,0.0000,1.500000,1.000000,3.000000,1.000000,32.000000,0.500000,141.000000,4.000000,...,564.000000,1247.000000,2146.500000,12.000000,5.000000,11.000000,1.000000,1.000000,1.000000,1.500000
75%,1615.250000,1.0000,2.200000,1.000000,7.000000,1.000000,48.000000,0.800000,170.000000,7.000000,...,947.250000,1633.000000,3064.500000,16.000000,9.000000,16.000000,1.000000,1.000000,1.000000,2.250000
max,1998.000000,1.0000,3.000000,1.000000,19.000000,1.000000,64.000000,1.000000,200.000000,8.000000,...,1960.000000,1998.000000,3998.000000,19.000000,18.000000,20.000000,1.000000,1.000000,1.000000,3.000000


# ⚖️ Normalization — কেন করতে হয়?

## 🧠 বিস্তারিত থিওরি

ধরো দুটি Feature:
- **RAM**: 256 – 3998
- **Clock Speed**: 0.5 – 3.0

ML Algorithm (বিশেষত Distance-based যেমন KNN, SVM) এই দুটি Feature দেখে মনে করবে RAM অনেক বেশি গুরুত্বপূর্ণ, কারণ এর সংখ্যা অনেক বড়। কিন্তু বাস্তবে Clock Speed হয়তো Price Range-এর উপর বেশি প্রভাব ফেলছে।

**Normalization** এই সমস্যার সমাধান করে — সব Feature-কে একটি নির্দিষ্ট Range-এ নিয়ে আসে।

### দুটি প্রধান পদ্ধতি:

| পদ্ধতি | Range | সূত্র | কখন ব্যবহার |
|---|---|---|---|
| **Min-Max Scaling** | 0 থেকে 1 | $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ | Outlier কম থাকলে |
| **Standard Scaler** | Mean=0, Std=1 | $x' = \frac{x - \mu}{\sigma}$ | Outlier বেশি থাকলে |

> ⚠️ **কখন Normalization করতে হয় না?** Decision Tree ও Random Forest এর মতো Tree-based Algorithm-এ Normalization ছাড়াও ভালো কাজ করে, কারণ এগুলো Distance-based নয়।

## 1️⃣ Min-Max Scaling

### 🧠 থিওরি

**Min-Max Scaling** প্রতিটি মানকে **0 থেকে 1** এর মধ্যে নিয়ে আসে।

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

- $x_{min}$ = সেই Column-এর সর্বনিম্ন মান
- $x_{max}$ = সেই Column-এর সর্বোচ্চ মান
- ফলাফল সবসময় **0 ≤ x' ≤ 1** এর মধ্যে থাকে

### সুবিধা:
- সব মান 0 থেকে 1 এর মধ্যে আসে — সহজে বোঝা যায়
- Neural Network ও Image Processing এ বহুল ব্যবহৃত

### সমস্যা:
- **Outlier** এর প্রতি Sensitive — একটি অস্বাভাবিক বড় মান পুরো Scaling নষ্ট করে দিতে পারে

> ⚠️ **সাধারণ ভুল:** `fit_transform()` দুটি কাজ একসাথে করে — প্রথমে `fit()` (min ও max শেখে) তারপর `transform()` (সেই মান দিয়ে Scale করে)। আলাদা আলাদা করে `fit()` ও `transform()` লেখাও যায়, কিন্তু **Test Data-তে কখনো `fit_transform()` ব্যবহার করতে হয় না** — শুধু `transform()` ব্যবহার করতে হয়, নাহলে Data Leakage হয়।

**Applying Min-Max Scaling** ✅

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
mn=MinMaxScaler()

In [12]:
mn.fit_transform(dataset)

array([[0.22778891, 0.        , 0.68      , ..., 0.        , 1.        ,
        0.33333333],
       [0.34736139, 1.        , 0.        , ..., 1.        , 0.        ,
        0.66666667],
       [0.04141617, 1.        , 0.        , ..., 1.        , 0.        ,
        0.66666667],
       ...,
       [0.94188377, 0.        , 0.16      , ..., 1.        , 0.        ,
        1.        ],
       [0.6753507 , 0.        , 0.16      , ..., 1.        , 1.        ,
        0.        ],
       [0.00601202, 1.        , 0.6       , ..., 1.        , 1.        ,
        1.        ]])

## 📋 Min-Max Scaled Data — DataFrame হিসেবে দেখা

`mn.fit_transform()` একটি **NumPy Array** রিটার্ন করে। `pd.DataFrame()` দিয়ে সেটিকে সুন্দর DataFrame-এ রূপান্তর করা হচ্ছে — Column-এর নামসহ দেখা যাবে।

In [13]:
pd.DataFrame(mn.fit_transform(dataset))

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.227789,0.0,0.68,0.0,0.052632,0.0,0.080645,0.555556,0.900000,0.142857,...,0.010204,0.170895,0.612774,0.285714,0.388889,0.944444,0.0,0.0,1.0,0.333333
1,0.347361,1.0,0.00,1.0,0.000000,1.0,0.822581,0.666667,0.466667,0.285714,...,0.461735,0.993324,0.634687,0.857143,0.166667,0.277778,1.0,1.0,0.0,0.666667
2,0.041416,1.0,0.00,1.0,0.105263,1.0,0.629032,0.888889,0.541667,0.571429,...,0.644388,0.811749,0.627205,0.428571,0.111111,0.388889,1.0,1.0,0.0,0.666667
3,0.076152,1.0,0.80,0.0,0.000000,0.0,0.129032,0.777778,0.425000,0.714286,...,0.620408,0.858478,0.671566,0.785714,0.444444,0.500000,1.0,0.0,0.0,0.666667
4,0.881764,1.0,0.28,0.0,0.684211,1.0,0.677419,0.555556,0.508333,0.142857,...,0.616327,0.475300,0.308658,0.214286,0.111111,0.722222,1.0,1.0,0.0,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.195725,1.0,0.00,1.0,0.000000,1.0,0.000000,0.777778,0.216667,0.714286,...,0.623469,0.927904,0.110102,0.571429,0.222222,0.944444,1.0,1.0,0.0,0.000000
1996,0.977956,1.0,0.84,1.0,0.000000,0.0,0.596774,0.111111,0.891667,0.428571,...,0.466837,0.977971,0.474613,0.428571,0.555556,0.777778,1.0,1.0,1.0,0.666667
1997,0.941884,0.0,0.16,1.0,0.052632,1.0,0.548387,0.666667,0.233333,1.000000,...,0.442857,0.755674,0.748530,0.285714,0.055556,0.166667,1.0,1.0,0.0,1.000000
1998,0.675351,0.0,0.16,0.0,0.210526,1.0,0.709677,0.000000,0.541667,0.571429,...,0.171429,0.113485,0.163816,0.928571,0.555556,0.944444,1.0,1.0,1.0,0.000000


## 2️⃣ Standard Scaler (Z-Score Normalization)

### 🧠 থিওরি

**Standard Scaler** প্রতিটি মানকে **Mean = 0, Standard Deviation = 1** এ রূপান্তর করে।

$$x' = \frac{x - \mu}{\sigma}$$

- $\mu$ = Column-এর গড় (Mean)
- $\sigma$ = Column-এর Standard Deviation
- ফলাফল **ঋণাত্মক (-) থেকে ধনাত্মক (+)** হতে পারে, কোনো নির্দিষ্ট Range নেই

### সুবিধা:
- **Outlier-এর প্রতি কম Sensitive** — Min-Max এর চেয়ে বেশি Robust
- Linear Regression, Logistic Regression, SVM এর জন্য আদর্শ

### Min-Max vs Standard Scaler — কোনটা বেছে নেবে?

| পরিস্থিতি | ব্যবহার করো |
|---|---|
| Outlier কম, Range 0-1 দরকার | **Min-Max Scaler** |
| Outlier আছে, Distribution Normal | **Standard Scaler** |
| Neural Network, Image Data | **Min-Max Scaler** |
| Linear/Logistic Regression, SVM | **Standard Scaler** |

**Standard Scaler** ✅

In [14]:
from sklearn.preprocessing import StandardScaler
sd=StandardScaler()

In [16]:
sd.fit_transform(dataset)

array([[-0.90259726, -0.9900495 ,  0.83077942, ..., -1.00601811,
         0.98609664, -0.4472136 ],
       [-0.49513857,  1.0100505 , -1.2530642 , ...,  0.99401789,
        -1.01409939,  0.4472136 ],
       [-1.5376865 ,  1.0100505 , -1.2530642 , ...,  0.99401789,
        -1.01409939,  0.4472136 ],
       ...,
       [ 1.53077336, -0.9900495 , -0.76274805, ...,  0.99401789,
        -1.01409939,  1.34164079],
       [ 0.62252745, -0.9900495 , -0.76274805, ...,  0.99401789,
         0.98609664, -1.34164079],
       [-1.65833069,  1.0100505 ,  0.58562134, ...,  0.99401789,
         0.98609664,  1.34164079]])

## 📋 Standard Scaled Data — DataFrame হিসেবে দেখা

এখানে মানগুলো 0 থেকে 1 এর মধ্যে নেই — ঋণাত্মক ও ধনাত্মক উভয় মান থাকতে পারে। এটাই Standard Scaler-এর বৈশিষ্ট্য।

In [17]:
pd.DataFrame(sd.fit_transform(dataset))

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,-0.902597,-0.990050,0.830779,-1.019184,-0.762495,-1.043966,-1.380644,0.340740,1.349249,-1.101971,...,-1.408949,-1.146784,0.391703,-0.784983,0.283103,1.462493,-1.786861,-1.006018,0.986097,-0.447214
1,-0.495139,1.010051,-1.253064,0.981177,-0.992890,0.957886,1.155024,0.687548,-0.120059,-0.664768,...,0.585778,1.704465,0.467317,1.114266,-0.635317,-0.734267,0.559641,0.994018,-1.014099,0.447214
2,-1.537686,1.010051,-1.253064,0.981177,-0.532099,0.957886,0.493546,1.381165,0.134244,0.209639,...,1.392684,1.074968,0.441498,-0.310171,-0.864922,-0.368140,0.559641,0.994018,-1.014099,0.447214
3,-1.419319,1.010051,1.198517,-1.019184,-0.992890,-1.043966,-1.215274,1.034357,-0.261339,0.646842,...,1.286750,1.236971,0.594569,0.876859,0.512708,-0.002014,0.559641,-1.006018,-1.014099,0.447214
4,1.325906,1.010051,-0.395011,-1.019184,2.002254,0.957886,0.658915,0.340740,0.021220,-1.101971,...,1.268718,-0.091452,-0.657666,-1.022389,-0.864922,0.730240,0.559641,0.994018,-1.014099,-0.447214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,-1.011860,1.010051,-1.253064,0.981177,-0.992890,0.957886,-1.656260,1.034357,-0.967737,0.646842,...,1.300273,1.477661,-1.342799,0.164641,-0.405712,1.462493,0.559641,0.994018,-1.014099,-1.341641
1996,1.653694,1.010051,1.321096,0.981177,-0.992890,-1.043966,0.383299,-1.046495,1.320993,-0.227564,...,0.608317,1.651235,-0.085031,-0.310171,0.971917,0.913303,0.559641,0.994018,0.986097,0.447214
1997,1.530773,-0.990050,-0.762748,0.981177,-0.762495,0.957886,0.217930,0.687548,-0.911225,1.521249,...,0.502383,0.880565,0.860139,-0.784983,-1.094526,-1.100394,0.559641,0.994018,-1.014099,1.341641
1998,0.622527,-0.990050,-0.762748,-1.019184,-0.071307,0.957886,0.769162,-1.393304,0.134244,0.209639,...,-0.696707,-1.345816,-1.157454,1.351672,0.971917,1.462493,0.559641,0.994018,0.986097,-1.341641


# 🔧 Feature Engineering — সঠিক Feature বাছাই

## 🧠 বিস্তারিত থিওরি

**Feature Engineering** হলো ML Pipeline-এর সবচেয়ে গুরুত্বপূর্ণ ধাপগুলোর একটি। এর মূল লক্ষ্য:

> "সব Feature সমান গুরুত্বপূর্ণ নয় — মডেলকে শুধু সবচেয়ে কার্যকর Feature দিয়ে Train করলে ভালো ফলাফল পাওয়া যায়।"

### কেন Feature Selection জরুরি?

| সমস্যা | Feature Selection ছাড়া | Feature Selection সহ |
|---|---|---|
| Training Speed | ধীর (বেশি Feature) | দ্রুত |
| Overfitting | বেশি | কম |
| Accuracy | কম হতে পারে | বেশি |
| Interpretability | কঠিন | সহজ |

### `iloc` দিয়ে Column Select করা

`iloc` হলো **Integer Location** — সংখ্যার Index দিয়ে Row ও Column Select করার পদ্ধতি।

| Code | অর্থ |
|---|---|
| `iloc[:, :1]` | সব Row, শুধু প্রথম Column |
| `iloc[:, :-1]` | সব Row, শেষ Column বাদ দিয়ে সব Column (X) |
| `iloc[:, -1]` | সব Row, শুধু শেষ Column (Y) |

**Feature Engineering** ✅

**Dependent Column more than 1 hole ei iloc use korbo** ✅

In [18]:
dataset.iloc[:,:-1]

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
0,842,0,2.2,0,1,0,7,0.6,188,2,2,20,756,2549,9,7,19,0,0,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,6,905,1988,2631,17,3,7,1,1,0
2,563,1,0.5,1,2,1,41,0.9,145,5,6,1263,1716,2603,11,2,9,1,1,0
3,615,1,2.5,0,0,0,10,0.8,131,6,9,1216,1786,2769,16,8,11,1,0,0
4,1821,1,1.2,0,13,1,44,0.6,141,2,14,1208,1212,1411,8,2,15,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,794,1,0.5,1,0,1,2,0.8,106,6,14,1222,1890,668,13,4,19,1,1,0
1996,1965,1,2.6,1,0,0,39,0.2,187,4,3,915,1965,2032,11,10,16,1,1,1
1997,1911,0,0.9,1,1,1,36,0.7,108,8,3,868,1632,3057,9,1,5,1,1,0
1998,1512,0,0.9,0,4,1,46,0.1,145,5,5,336,670,869,18,10,19,1,1,1


## 📥 X (Independent Features) তৈরি করা

`iloc[:, :-1]` → সব Row এবং **শেষ Column বাদ দিয়ে বাকি সব Column** — এগুলোই আমাদের Input Features (X)।

আমাদের Dataset-এ শেষ Column হলো `price_range` (যা Predict করতে চাই), তাই বাকি সব Column হলো Independent Variable।

In [19]:
x= dataset.iloc[:,:-1]

## 📤 Y (Dependent/Target Variable) তৈরি করা

`price_range` হলো আমাদের **Target Variable** — মোবাইলের Price কোন Category তে পড়ে (0=Low, 1=Medium, 2=High, 3=Very High) তা Predict করতে চাই।

দুটো উপায়ে Y তৈরি করা যায়:
- `iloc[:, -1]` → Position দিয়ে (শেষ Column)
- `dataset["price_range"]` → নাম দিয়ে

দুটোই একই ফলাফল দেবে।

In [20]:
y= dataset.iloc[:,-1]
y

,price_range
0,1
1,2
2,2
3,2
4,1
...,...
1995,0
1996,2
1997,3
1998,0


In [21]:
y=dataset["price_range"]
y

,price_range
0,1
1,2
2,2
3,2
4,1
...,...
1995,0
1996,2
1997,3
1998,0


# 📐 Chi-Square Test দিয়ে Best Feature Select করা

## 🧠 বিস্তারিত থিওরি

**`SelectKBest`** + **`chi2` (Chi-Square Test)** — এই দুটি মিলে কাজ করে সবচেয়ে গুরুত্বপূর্ণ Feature গুলো বেছে নিতে।

### Chi-Square Test কী?

**Chi-Square ($\chi^2$)** একটি Statistical Test যা দেখে প্রতিটি Independent Feature এবং Target Variable-এর মধ্যে কতটা **নির্ভরশীলতা (Dependency)** আছে।

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

- $O$ = Observed (আসল) মান
- $E$ = Expected (প্রত্যাশিত) মান
- **Score যত বেশি → Feature তত বেশি গুরুত্বপূর্ণ**

### কীভাবে কাজ করে?

1. প্রতিটি Feature-এর জন্য একটি Chi-Square Score বের করা হয়
2. Score যত বেশি, সেই Feature Target Variable-এর উপর তত বেশি প্রভাব রাখে
3. `SelectKBest` শুধু সবচেয়ে বেশি Score পাওয়া K টি Feature রাখে

> ⚠️ **গুরুত্বপূর্ণ:** Chi-Square Test শুধুমাত্র **Non-Negative (0 বা তার বেশি) Numerical Data-তে** কাজ করে। Negative মান থাকলে Error আসবে। তাই আগে Normalization (Min-Max) করে নেওয়া ভালো।

**21 ta feature er moddhe kon feature ta select korbe eita ashbe chi2 er result er upor. Eita score_function e store hobe erpr best feature select korbe SelectKBest er maddhome** ✅

In [22]:
from sklearn.feature_selection import SelectKBest, chi2


## ⚙️ SelectKBest Object তৈরি করা

`score_func=chi2` → বলছি যে Chi-Square পদ্ধতিতে প্রতিটি Feature-এর Score বের করো।

এই মুহূর্তে `sc` Object তৈরি হলো কিন্তু এখনো কোনো Data দেখেনি — পরের ধাপে `fit()` দিয়ে Data দেখাতে হবে।

In [23]:
sc=SelectKBest(score_func=chi2)

## 🏋️ Model Fit করা — `sc.fit(x, y)`

`fit(x, y)` দিলে SelectKBest প্রতিটি Feature (X Column) এবং Target (Y) এর মধ্যে Chi-Square Score বের করে।

> ⚠️ **সাধারণ ভুল:** `fit()` এ শুধু `x` দিলে Error আসবে — `y` অবশ্যই দিতে হবে, কারণ Score বের করতে হলে Target জানতেই হবে।

In [24]:
sc.fit(x,y)

SelectKBest(score_func=<function chi2 at 0x78be6674a0c0>)

## 📊 Feature Scores দেখা — `sc.scores_`

`sc.scores_` একটি Array যেখানে প্রতিটি Independent Column-এর **Chi-Square Score** দেওয়া আছে। যে Column-এর Score সবচেয়ে বেশি, সেটি Target Variable-এর উপর সবচেয়ে বেশি প্রভাব রাখে।

**Tar mane 20 ta independent column er score eikhane dekha jaitese** ✅

**Mane eikhane 13 number column sobcheye beshi important** ✅

In [25]:
sc.scores_


array([1.41298666e+04, 7.23232323e-01, 6.48365906e-01, 6.31010795e-01,
       1.01351665e+01, 1.52157239e+00, 8.98391244e+01, 7.45819631e-01,
       9.59728626e+01, 9.09755558e+00, 9.18605355e+00, 1.73635695e+04,
       9.81058675e+03, 9.31267519e+05, 9.61487832e+00, 1.64803191e+01,
       1.32364000e+01, 3.27642810e-01, 1.92842942e+00, 4.22090730e-01])

## 📋 Score গুলো DataFrame-এ দেখা

শুধু Array দেখলে কোন Score কোন Column-এর সেটা বোঝা কঠিন। `pd.DataFrame()` দিয়ে সুন্দর Table আকারে দেখা হচ্ছে।

In [26]:
pd.DataFrame(sc.scores_)

,0
0,14129.866576
1,0.723232
2,0.648366
3,0.631011
4,10.135166
5,1.521572
6,89.839124
7,0.745820
8,95.972863
9,9.097556


## 🔗 Column নাম ও Score একসাথে দেখা — `pd.concat()`

`pd.concat()` দিয়ে দুটি DataFrame পাশাপাশি (axis=1 মানে Column-wise) জুড়ে দেওয়া হচ্ছে:
- প্রথমটি → Column-এর নাম
- দ্বিতীয়টি → সেই Column-এর Chi-Square Score

এতে একটি সুন্দর **নাম + Score** Table তৈরি হবে।

> ⚠️ **সাধারণ ভুল:** `axis=0` দিলে নিচে জুড়বে (Row-wise), `axis=1` দিলে পাশে জুড়বে (Column-wise)। এখানে পাশাপাশি দরকার তাই `axis=1` ব্যবহার করতে হবে।

In [27]:
pd.concat([pd.DataFrame(x.columns),pd.DataFrame(sc.scores_)],axis=1)

,0,0
0,battery_power,14129.866576
1,blue,0.723232
2,clock_speed,0.648366
3,dual_sim,0.631011
4,fc,10.135166
5,four_g,1.521572
6,int_memory,89.839124
7,m_dep,0.745820
8,mobile_wt,95.972863
9,n_cores,9.097556


## 🏷️ Score Column-কে নাম দেওয়া

আগের কোডে Score Column-এর কোনো নাম ছিল না (শুধু `0` দেখাচ্ছিল)। এখন `columns=["Score"]` দিয়ে Column-কে সুন্দর নাম দেওয়া হচ্ছে এবং `pc` ভ্যারিয়েবলে Store করা হচ্ছে পরে ব্যবহারের জন্য।

In [30]:
pc= pd.concat([pd.DataFrame(x.columns,columns=["Feature"]),pd.DataFrame(sc.scores_,columns=["Score"])],axis=1)

In [31]:
pc

,Feature,Score
0,battery_power,14129.866576
1,blue,0.723232
2,clock_speed,0.648366
3,dual_sim,0.631011
4,fc,10.135166
5,four_g,1.521572
6,int_memory,89.839124
7,m_dep,0.745820
8,mobile_wt,95.972863
9,n_cores,9.097556


## 🏆 সেরা ৮টি Feature বের করা — `nlargest()`

`nlargest(8, "Score")` → Score Column-এ সবচেয়ে বেশি মানের **শীর্ষ ৮টি Row** বের করে দেখাবে — অর্থাৎ **সবচেয়ে গুরুত্বপূর্ণ ৮টি Feature**।

> 💡 **পরের ধাপ:** এই ৮টি Feature নিয়ে `x` নতুনভাবে তৈরি করে ML Model Train করলে আগের চেয়ে ভালো Accuracy পাওয়া যেতে পারে — কারণ অপ্রয়োজনীয় Feature বাদ দেওয়া হয়েছে।

---

# 🏁 চূড়ান্ত সারসংক্ষেপ (Final Summary)

| বিষয় | কী শিখলাম |
|---|---|
| **Min-Max Scaler** | সব মান 0–1 এ নিয়ে আসে, Outlier-sensitive |
| **Standard Scaler** | Mean=0, Std=1, Outlier-robust |
| **`iloc[:,:-1]`** | শেষ Column বাদ দিয়ে সব (X) |
| **`iloc[:,-1]`** | শুধু শেষ Column (Y) |
| **Chi-Square Test** | Feature ও Target এর Dependency Score বের করে |
| **`SelectKBest`** | সর্বোচ্চ Score-ওয়ালা K টি Feature রাখে |
| **`nlargest()`** | Score অনুযায়ী শীর্ষ N Feature দেখায় |

> ⭐ যদি এই নোটবুকটি ভালো লাগে, GitHub-এ Star দিতে ভুলো না!

In [32]:
pc.nlargest(8,"Score")

,Feature,Score
13,ram,931267.519053
11,px_height,17363.569536
0,battery_power,14129.866576
12,px_width,9810.586750
8,mobile_wt,95.972863
6,int_memory,89.839124
15,sc_w,16.480319
16,talk_time,13.236400
